## Assignment: Student Management Database in PostgreSQL
Objective
In this exercise, you will connect to a PostgreSQL database you created, build the tables for a simple student management system, insert sample data, and perform different queries — including JOIN operations.

Part 1: Database Connection
Create a new database in PostgreSQL named student_management.

Write Python code (using psycopg2) to connect to your student_management database.

Example:

In [3]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="student_management",
    user="postgres",
    password="0000"
)

cur = conn.cursor()
print("Database connected successfully ")


Database connected successfully 


## Part 2: Create Tables

Create the following tables using SQL:

### `students`
- `student_id` (SERIAL PRIMARY KEY)  
- `name` (VARCHAR)  
- `email` (VARCHAR)  
- `phone` (VARCHAR)  

### `courses`
- `course_id` (SERIAL PRIMARY KEY)  
- `course_name` (VARCHAR)  
- `credits` (INT)  

### `enrollments`
- `enrollment_id` (SERIAL PRIMARY KEY)  
- `student_id` (INT REFERENCES students(student_id))  
- `course_id` (INT REFERENCES courses(course_id))  
- `grade` (VARCHAR)  


In [4]:
cur.execute("""
CREATE TABLE students (
    student_id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100),
    phone VARCHAR(20)
);
""")
print("Table created successfully ")

cur.execute("""
CREATE TABLE courses (
    course_id SERIAL PRIMARY KEY,
    course_name VARCHAR(100),
    credits INT
);
""")
print("Table created successfully ")

cur.execute("""
CREATE TABLE enrollments (
    enrollment_id SERIAL PRIMARY KEY,
    student_id INT REFERENCES students(student_id),
    course_id INT REFERENCES courses(course_id),
    grade VARCHAR(2)
);
""")

conn.commit()
print("Tables created successfully ")


Table created successfully 
Table created successfully 
Tables created successfully 


## Part 3: Insert Sample Data
Insert at least 3 records into each table.

Example:

In [5]:
# Insert students
cur.execute("INSERT INTO students (name, email, phone) VALUES (%s,%s,%s)",
            ("Mina", "mina@mail.com", "01011111111"))

cur.execute("INSERT INTO students (name, email, phone) VALUES (%s,%s,%s)",
            ("Sara", "sara@mail.com", "01022222222"))

# Insert courses
cur.execute("INSERT INTO courses (course_name, credits) VALUES (%s,%s)",
            ("Database", 3))

cur.execute("INSERT INTO courses (course_name, credits) VALUES (%s,%s)",
            ("Python", 4))

conn.commit()
print("Data inserted successfully ")


Data inserted successfully 


In [35]:
# Insert enrollments
cur.execute("INSERT INTO enrollments (student_id, course_id, grade) VALUES (%s,%s,%s);", (1, 1, 'A'))  # Mina - Database
cur.execute("INSERT INTO enrollments (student_id, course_id, grade) VALUES (%s,%s,%s);", (1, 2, 'B'))  # Mina - Python
cur.execute("INSERT INTO enrollments (student_id, course_id, grade) VALUES (%s,%s,%s);", (2, 2, 'A'))  # Sara - Python

conn.commit()
print("Enrollments inserted successfully ")


Enrollments inserted successfully 


In [36]:
cur.execute("SELECT * FROM enrollments;")
print(cur.fetchall())


[(1, 1, 1, 'A'), (2, 1, 2, 'B'), (3, 2, 2, 'A')]


In [9]:
def run_query(sql, params=None):
    cur.execute(sql, params or ())
    try:
        rows = cur.fetchall()
        for r in rows:
            print(r)
    except psycopg2.ProgrammingError:
        conn.commit()
        print("Done ")


## Part 4: Query Tasks

Write SQL queries to answer the following questions:

1. **List all students.**
   ```sql
   SELECT * FROM students;


In [10]:
run_query("SELECT * FROM students;")


(1, 'Mina', 'mina@mail.com', '01011111111')
(2, 'Sara', 'sara@mail.com', '01022222222')


2. Find students who have a grade of 'A'.
Hint: Use a WHERE clause in the enrollments table to filter by grade.

SELECT * 
FROM enrollments
WHERE grade = 'A';


In [12]:
run_query("""
SELECT DISTINCT s.*
FROM students s
JOIN enrollments e ON e.student_id = s.student_id
WHERE e.grade = 'A';
""")


In [15]:
conn.rollback()
print("rollback done successfully ")


rollback done successfully 


3. Show all courses with their credit hours.
Hint: Select course name and credit hours from the courses table.


SELECT course_name, credit_hours 
FROM courses;

In [24]:
conn.rollback()
cur = conn.cursor()

cur.execute("SELECT course_name, credits FROM courses;")
for row in cur.fetchall():
    print(row)


('Database', 3)
('Python', 4)


4. Find the courses a specific student (e.g., 'Alice Johnson') is enrolled in.
Hint: Use JOIN between students, enrollments, and courses.


SELECT s.name, c.course_name
FROM students s
JOIN enrollments e ON s.student_id = e.student_id
JOIN courses c ON e.course_id = c.course_id
WHERE s.name = 'Alice Johnson';

In [37]:
cur.execute("""
SELECT s.name, c.course_name
FROM students s
JOIN enrollments e ON s.student_id = e.student_id
JOIN courses c ON e.course_id = c.course_id
WHERE s.name = 'Mina';
""")
print(cur.fetchall())


[('Mina', 'Database'), ('Mina', 'Python')]


5. List each student along with the number of courses they are enrolled in.
Hint: Use JOIN, GROUP BY, and COUNT().


SELECT s.name, COUNT(e.course_id) AS total_courses
FROM students s
JOIN enrollments e ON s.student_id = e.student_id
GROUP BY s.name;

In [38]:
cur.execute("""
SELECT s.name, COUNT(e.course_id) AS total_courses
FROM students s
LEFT JOIN enrollments e ON s.student_id = e.student_id
GROUP BY s.name
ORDER BY total_courses DESC;
""")
print(cur.fetchall())


[('Mina', 2), ('Sara', 1)]


6. Find students who are not enrolled in any course.
Hint: Use LEFT JOIN and check for NULL in the enrollments table.


SELECT s.name
FROM students s
LEFT JOIN enrollments e ON s.student_id = e.student_id
WHERE e.course_id IS NULL;


In [39]:
cur.execute("""
SELECT s.name
FROM students s
LEFT JOIN enrollments e ON s.student_id = e.student_id
WHERE e.course_id IS NULL;
""")
print(cur.fetchall())


[]


7. Show all enrollments sorted by grade (highest to lowest).
Hint: Use ORDER BY with DESC.


SELECT * 
FROM enrollments
ORDER BY grade DESC;



In [40]:
cur.execute("""
SELECT *
FROM enrollments
ORDER BY CASE grade
  WHEN 'A' THEN 1
  WHEN 'B' THEN 2
  WHEN 'C' THEN 3
  WHEN 'D' THEN 4
  WHEN 'F' THEN 5
  ELSE 6
END;
""")
print(cur.fetchall())


[(1, 1, 1, 'A'), (3, 2, 2, 'A'), (2, 1, 2, 'B')]


In [61]:
conn.rollback()
print("rollback ")


rollback 


Part 5: Bonus
Write a query to find the average grade per course (Convert grades into numeric values if you want to calculate an average).

Add a instructor table and modify your queries to include instructor names.



In [58]:
cur.execute("""
SELECT c.course_name,
       ROUND(AVG(CASE e.grade
           WHEN 'A' THEN 4
           WHEN 'B' THEN 3
           WHEN 'C' THEN 2
           WHEN 'D' THEN 1
           WHEN 'F' THEN 0
           ELSE NULL
       END)::numeric, 2) AS avg_grade_points
FROM courses c
JOIN enrollments e ON e.course_id = c.course_id
GROUP BY c.course_name
ORDER BY avg_grade_points DESC;
""")
print(cur.fetchall())


[('Database', Decimal('4.00')), ('Python', Decimal('3.50'))]


In [43]:
cur.execute("""
CREATE TABLE IF NOT EXISTS instructors (
    instructor_id SERIAL PRIMARY KEY,
    instructor_name VARCHAR(100) NOT NULL
);
""")
conn.commit()
print("instructors table created ")


instructors table created 


In [44]:
cur.execute("""
ALTER TABLE courses
ADD COLUMN IF NOT EXISTS instructor_id INT REFERENCES instructors(instructor_id);
""")
conn.commit()
print("instructor_id added to courses table ")


instructor_id added to courses table 


In [45]:
cur.execute("INSERT INTO instructors (instructor_name) VALUES ('Dr. Ahmed');")
cur.execute("INSERT INTO instructors (instructor_name) VALUES ('Dr. Mona');")
conn.commit()
print("instructors inserted ")


instructors inserted 


In [46]:
cur.execute("UPDATE courses SET instructor_id = 1 WHERE course_id = 1;")  # Database -> Dr. Ahmed
cur.execute("UPDATE courses SET instructor_id = 2 WHERE course_id = 2;")  # Python   -> Dr. Mona
conn.commit()
print("courses linked to instructors successfully ")


courses linked to instructors successfully 


In [47]:
cur.execute("""
SELECT c.course_name, i.instructor_name
FROM courses c
JOIN instructors i ON i.instructor_id = c.instructor_id;
""")
print(cur.fetchall())


[('Database', 'Dr. Ahmed'), ('Python', 'Dr. Mona')]


In [70]:
conn.rollback()
print("rollback done ")


rollback done 


In [71]:
cur.execute("""
SELECT c.course_name,
       i.instructor_name,
       AVG(CASE e.grade
           WHEN 'A' THEN 4
           WHEN 'B' THEN 3
           WHEN 'C' THEN 2
           WHEN 'D' THEN 1
           WHEN 'F' THEN 0
           ELSE NULL
       END) AS avg_grade_points
FROM courses c
JOIN instructors i ON i.instructor_id = c.instructor_id
JOIN enrollments e ON e.course_id = c.course_id
GROUP BY c.course_name, i.instructor_name
ORDER BY avg_grade_points DESC;
""")
print(cur.fetchall())


[('Database', 'Dr. Ahmed', Decimal('4.0000000000000000')), ('Python', 'Dr. Mona', Decimal('3.5000000000000000'))]
